# Academy Review Pipeline

This notebook runs academy review checks component by component. Configure test/full mode once in the setup cell, then run each component cell you want included in the final workbook. The notebook stays thin; extraction, deterministic checks, optional OpenAI semantic review, caching, and workbook generation live in `sea_pipeline.review`.

What you can run here:
- Validate paths and credentials.
- Analyze takeaways, concepts, resources, quizzes, polls, and case studies in separate cells.
- Write the combined review workbook after the component cells have run.
- Print a simple category completion diagram from the generated workbook.
- Run spellcheck audit from the review workflow.


In [ ]:
from pathlib import Path

from sea_pipeline.shared.config import PipelineConfig
from sea_pipeline.shared.logging import print_stage_result
from sea_pipeline.review.runner import (
    create_review_session,
    list_review_stages,
    run_review_component,
    run_review_stage,
    write_review_session_workbook,
)
from sea_pipeline.review.report import print_review_completion_summary

config = PipelineConfig.from_repo_root()

# Set this once for the run. Positive integer reviews a sequential lesson slice; False scans all lessons.
REVIEW_TEST_START_LESSON_ID = "1.1.1"
REVIEW_TEST_LESSON_COUNT = False  # Set to False for full run

# AI review mode: False/off for deterministic only, True/full for full semantic review, or "Critical" for severe issues only.
REVIEW_RUN_AI_REVIEW = False
REVIEW_CHECK_RESOURCE_BLOBS = True
REVIEW_KEEP_PASS_ROWS = False
REVIEW_CONTROLS_PATH = config.paths.code_dir / "sea_pipeline" / "review" / "review_controls.csv"

# Outputs and cache. The cache prevents re-reviewing unchanged semantic checks with OpenAI.
REVIEW_OUTPUT_PATH = config.paths.outputs_dir / "Review" / "academy_review.xlsx"
REVIEW_CACHE_PATH = config.paths.outputs_dir / "Review" / "review_cache.json"
REVIEW_LESSON_ROOT = config.paths.outputs_dir / "SEA_Modules" / "en"

config.stage_options.update({
    "review_test_start_lesson_id": REVIEW_TEST_START_LESSON_ID,
    "review_test_lesson_count": REVIEW_TEST_LESSON_COUNT,
    "review_run_ai_review": REVIEW_RUN_AI_REVIEW,
    "review_check_resource_blobs": REVIEW_CHECK_RESOURCE_BLOBS,
    "review_keep_pass": REVIEW_KEEP_PASS_ROWS,
    "review_controls_path": REVIEW_CONTROLS_PATH,
    "review_output_path": REVIEW_OUTPUT_PATH,
    "review_cache_path": REVIEW_CACHE_PATH,
    "review_lesson_root": REVIEW_LESSON_ROOT,
})

print("Review stages:", ", ".join(list_review_stages()))
print("Mode:", "test" if REVIEW_TEST_LESSON_COUNT else "full")
print("Test slice:", REVIEW_TEST_START_LESSON_ID, "count", REVIEW_TEST_LESSON_COUNT if REVIEW_TEST_LESSON_COUNT else "all")
print("AI semantic review:", REVIEW_RUN_AI_REVIEW if REVIEW_RUN_AI_REVIEW else "off")
print("Keep pass rows:", REVIEW_KEEP_PASS_ROWS)
print("Review controls:", REVIEW_CONTROLS_PATH)
print("Review output:", REVIEW_OUTPUT_PATH)

# The session holds loaded lessons and accumulated component results for this notebook run.
review_session = create_review_session(config)


## Validate Environment

Run this first. It checks lesson paths and required credentials for the configured options, including Azure OpenAI and resource blob checks when enabled.


In [ ]:
result = run_review_stage(config, "validate_env")
print_stage_result(result)

## Analyze Takeaways

Checks the three lesson takeaways for count, placement near the end, title/body word counts, duplicates, and optional semantic quality.


In [ ]:
result = run_review_component(review_session, "takeaways")
print_stage_result(result)

## Analyze Key Concepts

Checks the three key concepts for glossary-style titles, clear definitions, word counts, and duplicates.


In [ ]:
result = run_review_component(review_session, "concepts")
print_stage_result(result)

## Analyze Resources

Checks resource count, text/href/image fields, URL validity, duplicates, and whether resource images exist in blob storage when enabled.


In [ ]:
result = run_review_component(review_session, "resources")
print_stage_result(result)

## Analyze Quizzes

Checks module quizzes for minimum/preferred question counts, prompt/options/solution/feedback completeness, and duplicates.


In [ ]:
result = run_review_component(review_session, "quiz")
print_stage_result(result)

## Analyze Polls

Checks poll presence, title, options, labels, and duplicate options.


In [ ]:
result = run_review_component(review_session, "poll")
print_stage_result(result)

## Analyze Case Studies

Checks case-study covers and the following content for title/body completeness and duplicate titles.


In [ ]:
result = run_review_component(review_session, "case_study")
print_stage_result(result)

## Spellcheck Audit

Run this when you want spelling and text-quality issues included as part of the review workflow. This delegates to the existing auxiliary spellcheck implementation and writes its CSV report to the established output path.


In [ ]:
result = run_review_stage(config, "spellcheck_audit")
print_stage_result(result)

## Write Review Workbook

Run this after the component cells you want included. It writes the combined workbook using the accumulated session results.


In [ ]:
result = write_review_session_workbook(review_session)
print_stage_result(result)

## Review Completion Summary

Run this after the workbook is generated. It reads `REVIEW_OUTPUT_PATH` and prints simple completion bars showing pass/fail completion by review category.


In [ ]:
print_review_completion_summary(REVIEW_OUTPUT_PATH)